## 2.4 Data Extraction and Web Scraping Assignment

### Question 1: The scraping of `https://www.scrapethissite.com/pages/forms/` in the last section assumes a hardcoded (fixed) no of pages. Can you improve the code by removing the hardcoded no of pages and instead use the `»` button to determine if there are more pages to scrape? Hint: Use a `while` loop.

```python
def parse_and_extract_rows(soup: BeautifulSoup):
    """
    Extract table rows from the parsed HTML.

    Args:
        soup: The parsed HTML.

    Returns:
        An iterator of dictionaries with the data from the current page.
    """
    header = soup.find('tr')
    headers = [th.text.strip() for th in header.find_all('th')]
    teams = soup.find_all('tr', 'team')
    for team in teams:
        row_dict = {}
        for header, col in zip(headers, team.find_all('td')):
            row_dict[header] = col.text.strip()
        yield row_dict
```

### Answer for Question 1: 

In [1]:
import requests # to make HTTP requests
import time # to add delays between requests
from bs4 import BeautifulSoup # to parse HTML

def parse_and_extract_rows(soup: BeautifulSoup):
    """
    Extract table rows from the parsed HTML.
    Args:
        soup: The parsed HTML.
    Returns:
        An iterator of dictionaries with the data from the current page.
    """
    header = soup.find('tr')
    headers = [th.text.strip() for th in header.find_all('th')]
    teams = soup.find_all('tr', 'team')
    for team in teams:
        row_dict = {}
        for header, col in zip(headers, team.find_all('td')):
            row_dict[header] = col.text.strip()
        yield row_dict 

In [2]:
r = requests.get("https://www.scrapethissite.com/pages/forms/") #make the HTTP request

In [3]:
r.status_code #to check if the request was successful

200

In [4]:
rows = []
page = 1

r = requests.get(f"https://www.scrapethissite.com/pages/forms/?page_num={page}")
soup = BeautifulSoup(r.text, "html.parser")
for row_dict in parse_and_extract_rows(soup):
    rows.append(row_dict)

while soup.find("a", {"aria-label": "Next"}):
    page += 1
    r = requests.get(f"https://www.scrapethissite.com/pages/forms/?page_num={page}")
    soup = BeautifulSoup(r.text, "html.parser")
    for row_dict in parse_and_extract_rows(soup):
        rows.append(row_dict)

    # pause for 1 second between requests
    time.sleep(1) # to be polite to the server

In [5]:
len(rows)

582

In [6]:
rows[0] # check the first row

{'Team Name': 'Boston Bruins',
 'Year': '1990',
 'Wins': '44',
 'Losses': '24',
 'OT Losses': '',
 'Win %': '0.55',
 'Goals For (GF)': '299',
 'Goals Against (GA)': '264',
 '+ / -': '35'}

In [7]:
rows[-1] # check the last row to confirm we have 582 rows (0-581)

{'Team Name': 'Winnipeg Jets',
 'Year': '2011',
 'Wins': '37',
 'Losses': '35',
 'OT Losses': '10',
 'Win %': '0.451',
 'Goals For (GF)': '225',
 'Goals Against (GA)': '246',
 '+ / -': '-21'}

In [8]:
rows[581] # check the last row to confirm we have 582 rows (0-581)

{'Team Name': 'Winnipeg Jets',
 'Year': '2011',
 'Wins': '37',
 'Losses': '35',
 'OT Losses': '10',
 'Win %': '0.451',
 'Goals For (GF)': '225',
 'Goals Against (GA)': '246',
 '+ / -': '-21'}

In [9]:
import pandas as pd
df = pd.DataFrame(rows) # create a DataFrame from the list of dictionaries

In [10]:
df = df.replace('', pd.NA).astype({'Year': int, 'Wins': 'Int64', 'Losses': 'Int64', 'OT Losses': 'Int64', 
                                   'Goals For (GF)': 'Int64', 'Goals Against (GA)': 'Int64', 
                                   '+ / -': 'Int64', 'Win %': float}) # convert columns to appropriate data types

In [11]:
df.info() # check the DataFrame info

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 582 entries, 0 to 581
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Team Name           582 non-null    object 
 1   Year                582 non-null    int64  
 2   Wins                582 non-null    Int64  
 3   Losses              582 non-null    Int64  
 4   OT Losses           358 non-null    Int64  
 5   Win %               582 non-null    float64
 6   Goals For (GF)      582 non-null    Int64  
 7   Goals Against (GA)  582 non-null    Int64  
 8   + / -               582 non-null    Int64  
dtypes: Int64(6), float64(1), int64(1), object(1)
memory usage: 44.5+ KB


In [12]:
df.head() # check the first few rows of the DataFrame

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,<NA>,0.550,299,264,35
1,Buffalo Sabres,1990,31,30,<NA>,0.388,292,278,14
2,Calgary Flames,1990,46,26,<NA>,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,<NA>,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,<NA>,0.425,273,298,-25


In [13]:
df.tail() # check the last few rows of the DataFrame

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
577,Tampa Bay Lightning,2011,38,36,8,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9,0.622,249,198,51
580,Washington Capitals,2011,42,32,8,0.512,222,230,-8
581,Winnipeg Jets,2011,37,35,10,0.451,225,246,-21


In [14]:
df

,Team Name,Year,Wins,Losses,OT Losses,Win %,Goals For (GF),Goals Against (GA),+ / -
0,Boston Bruins,1990,44,24,<NA>,0.550,299,264,35
1,Buffalo Sabres,1990,31,30,<NA>,0.388,292,278,14
2,Calgary Flames,1990,46,26,<NA>,0.575,344,263,81
3,Chicago Blackhawks,1990,49,23,<NA>,0.613,284,211,73
4,Detroit Red Wings,1990,34,38,<NA>,0.425,273,298,-25
...,...,...,...,...,...,...,...,...,...
577,Tampa Bay Lightning,2011,38,36,8,0.463,235,281,-46
578,Toronto Maple Leafs,2011,35,37,10,0.427,231,264,-33
579,Vancouver Canucks,2011,51,22,9,0.622,249,198,51
580,Washington Capitals,2011,42,32,8,0.512,222,230,-8
